# YeastCaduceus — Phase 0: Environment & Smoke Test

**Purpose:** Confirm PlantCAD2-Small loads, tokenizes 8192bp, runs a clean forward pass, LoRA adapter configs are readable, and Shorkie data is intact on Drive.

**Run order:** Cells must be run top to bottom in a fresh Colab session.

**Hardware:** Colab Pro A100 (CUDA 12.8, Python 3.12)

---

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 1 — Install
# 4-step order is mandatory. Do NOT collapse into one pip call.
#
# Why this order:
#   mamba-ssm 2.2.2 has a prebuilt wheel for torch 2.3.1+cu121 only.
#   It must be built against transformers==4.40.0 first.
#   causal-conv1d is a SEPARATE CUDA kernel package that Mamba2 requires at
#   runtime — omitting it causes:
#     AttributeError: 'NoneType' object has no attribute 'causal_conv1d_fwd'
#   transformers and peft are upgraded AFTER mamba-ssm is compiled.
#   peft==0.14.0 is required for eva_config in PlantCAD2 LoRA adapter_config.json.
# ─────────────────────────────────────────────────────────────────────────────

# Step 1 — PyTorch pinned to cu121 (overrides Colab default cu128)
!pip3 install torch==2.3.1 torchvision==0.18.1 torchaudio==2.3.1 \
    --index-url https://download.pytorch.org/whl/cu121 -q

# Step 2 — mamba-ssm prebuilt wheel; transformers==4.40.0 is a temporary pin
!pip3 install mamba-ssm==2.2.2 transformers==4.40.0 -q

# Step 2a — causal-conv1d: separate Mamba2 CUDA kernel (MUST follow mamba-ssm)
!pip3 install causal-conv1d==1.4.0 -q

# Step 3 — Upgrade transformers + peft after mamba-ssm/causal-conv1d are built
!pip3 install transformers==4.46.3 peft==0.14.0 -q

# Step 4 — Remaining deps
!pip3 install biopython pyfaidx pyBigWig datasets \
    scikit-learn scipy pandas numpy tqdm huggingface_hub -q

# ── Version verification ──────────────────────────────────────────────────────
import subprocess
expected = {
    'torch':        '2.3.1+cu121',
    'mamba_ssm':    '2.2.2',
    'causal_conv1d':'1.4.0',
    'transformers': '4.46.3',
    'peft':         '0.14.0',
}
print('Package versions:')
all_ok = True
for pkg, exp in expected.items():
    result = subprocess.run(['pip', 'show', pkg.replace('_', '-')],
                            capture_output=True, text=True)
    ver = next((l.split(': ')[1] for l in result.stdout.splitlines()
                if l.startswith('Version')), 'NOT FOUND')
    ok = '✅' if exp in ver else '❌'
    if '❌' in ok:
        all_ok = False
    print(f'  {ok} {pkg}: {ver} (expected: {exp})')

# sentence-transformers conflict warning on transformers==4.40.0 is benign — ignore it
print(f"\n{'✅ All versions correct' if all_ok else '❌ Version mismatch — check above'}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.9/780.9 MB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 64.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 124.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 120.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 63.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 133.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 731.7/731.7 MB 1.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 21.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 48.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.2/124.2 MB 22.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.0/196.0 MB 8.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 2 — Smoke Test: PlantCAD2-Small forward pass
#
# Key lessons learned:
#   1. CaduceusForMaskedLM is a pure SSM — no attention mechanism.
#      forward() accepts ONLY input_ids. Passing attention_mask or
#      token_type_ids raises TypeError. Always extract input_ids explicitly.
#   2. Actual param count is 176M (README says 88M — README is wrong).
#   3. d0768 = per-strand embedding size. BiMamba concatenates fwd + rev
#      strands: 768 + 768 = 1536 total hidden dim.
#      RC-equivariant avg: split 1536 → two 768s, flip RC, average → 768.
# ─────────────────────────────────────────────────────────────────────────────

from transformers import AutoTokenizer, AutoModelForMaskedLM
import torch

MODEL_ID = 'kuleshov-group/PlantCAD2-Small-l24-d0768'

print('Loading tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)

print('Loading model...')
model = AutoModelForMaskedLM.from_pretrained(MODEL_ID, trust_remote_code=True)
model.eval().to('cuda:0')

print()
print(f'Param count        : {sum(p.numel() for p in model.parameters())/1e6:.1f}M')
print(f'Vocab size         : {tokenizer.vocab_size}')
print(f'VRAM after load    : {torch.cuda.memory_allocated()/1e9:.2f} GB')

# ── Forward pass: full 8192bp window ─────────────────────────────────────────
seq = 'ACGT' * 2048  # exactly 8192bp

# IMPORTANT: extract only input_ids — Caduceus forward() rejects everything else
input_ids = tokenizer(seq, return_tensors='pt')['input_ids'].to('cuda:0')
print(f'\nInput shape        : {input_ids.shape}')  # [1, 8192]

with torch.no_grad():
    out = model(input_ids=input_ids, output_hidden_states=True)

hs = out.hidden_states[-1]
print(f'Hidden state shape : {hs.shape}')  # [1, 8192, 1536]

# ── RC-equivariant embedding averaging ───────────────────────────────────────
# d0768 = per-strand embedding. BiMamba cat fwd + rev → 1536 total.
# First 768 = forward strand, last 768 = reverse complement.
# Flip RC half along last dim before averaging → strand-invariant 768-dim embedding.
hidden_size = hs.shape[-1] // 2        # 768
fwd = hs[..., :hidden_size]            # [1, 8192, 768]
rev = hs[..., hidden_size:].flip(-1)   # [1, 8192, 768] flipped
avg = (fwd + rev) / 2                  # [1, 8192, 768]
print(f'Averaged embedding : {avg.shape}')
print(f'VRAM after forward : {torch.cuda.memory_allocated()/1e9:.2f} GB')

# ── Go / No-Go ────────────────────────────────────────────────────────────────
assert hs.shape  == (1, 8192, 1536), f'Unexpected hidden shape: {hs.shape}'
assert avg.shape == (1, 8192, 768),  f'Unexpected avg shape: {avg.shape}'
assert torch.cuda.memory_allocated()/1e9 < 10.0, 'VRAM too high for A100 training'

print('\n✅ Cell 2 PASSED — PlantCAD2-Small forward pass confirmed')

Loading tokenizer...
Loading model...

Param count        : 176.0M
Vocab size         : 7
VRAM after load    : 2.02 GB

Input shape        : torch.Size([1, 8192])
Hidden state shape : torch.Size([1, 8192, 1536])
Averaged embedding : torch.Size([1, 8192, 768])
VRAM after forward : 2.02 GB

✅ Cell 2 PASSED — PlantCAD2-Small forward pass confirmed


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 3 — LoRA target module discovery
#
# Goal: find which linear projection layers to inject LoRA adapters into.
# Strategy:
#   1. Try known/plausible Kuleshov HF repo names for published LoRA configs
#   2. If all are private/404, inspect PlantCAD2-Small's own named modules
#      and identify valid LoRA targets from the Mamba2 block structure
#   3. Cross-check against what the PlantCAD2 paper states:
#      "inserts rank-decomposition matrices into the feedforward layers"
# ─────────────────────────────────────────────────────────────────────────────

from huggingface_hub import hf_hub_download
import json

# ── Attempt 1: Try plausible public repo names ────────────────────────────────
# Replace the candidate_repos list in Cell 3 with the actual names:
candidate_repos = [
    'plantcad/cell_type_specific_acr_plantcad2_small',
    'plantcad/cross_species_acr_train_on_arabidopsis_plantcad2_small',
    'plantcad/cross_species_leaf_absolute_expression_plantcad2_small',
]

lora_configs = {}
print('Attempting to fetch published LoRA configs...')
for repo_id in candidate_repos:
    try:
        path = hf_hub_download(repo_id=repo_id, filename='adapter_config.json',
                               local_files_only=False)
        with open(path) as f:
            cfg = json.load(f)
        lora_configs[repo_id] = cfg
        print(f'  ✅ {repo_id}')
        for k in ['r', 'lora_alpha', 'lora_dropout', 'target_modules', 'bias', 'task_type']:
            if k in cfg:
                print(f'     {k}: {cfg[k]}')
    except Exception:
        print(f'  ⚠️  {repo_id} — not accessible (private or wrong name)')

# ── Attempt 2: Inspect model architecture directly ────────────────────────────
# The model is already loaded in memory from Cell 2
print(f'\n{"="*60}')
print('MAMBA2 BLOCK — LINEAR PROJECTION LAYERS (valid LoRA targets)')
print('(Paper states: LoRA inserted into feedforward/projection layers)')
print()

# Walk all named modules, find Linear layers inside Mamba blocks
from torch import nn
lora_candidates = []
for name, module in model.named_modules():
    if isinstance(module, nn.Linear):
        # Focus on Mamba block internals — ignore embeddings and LM head
        if any(k in name for k in ['in_proj', 'out_proj', 'x_proj', 'dt_proj',
                                    'A_log', 'D', 'norm', 'mixer']):
            lora_candidates.append((name, module.weight.shape))

# Group by layer type (strip the layer index)
import re
type_counts = {}
for name, shape in lora_candidates:
    # Get the leaf module name (e.g. 'in_proj', 'out_proj')
    leaf = name.split('.')[-1]
    if leaf not in type_counts:
        type_counts[leaf] = {'count': 0, 'shape_example': shape}
    type_counts[leaf]['count'] += 1

print(f'{"Layer type":<20} {"Count":>6}  {"Example shape"}')
print('-' * 50)
for leaf, info in sorted(type_counts.items()):
    print(f'  {leaf:<18} {info["count"]:>6}  {info["shape_example"]}')

# ── Recommendation ────────────────────────────────────────────────────────────
print(f'\n{"="*60}')
print('RECOMMENDED LoRA CONFIG for YeastCaduceus')
print()
if lora_configs:
    # Use fetched config
    cfg = next(iter(lora_configs.values()))
    print(f'  Source: fetched from Kuleshov HF repo')
    print(f'  r              : {cfg.get("r")}')
    print(f'  lora_alpha     : {cfg.get("lora_alpha")}')
    print(f'  lora_dropout   : {cfg.get("lora_dropout")}')
    print(f'  target_modules : {cfg.get("target_modules")}')
else:
    # Derive from architecture + paper description
    # Paper: "feedforward layers" in Mamba2 = in_proj and out_proj
    # These are the largest projection matrices in each Mamba block
    print('  Source: derived from Mamba2 architecture + PlantCAD2 paper')
    print('  r              : 8   (standard starting point; increase if underfits)')
    print('  lora_alpha     : 16  (2x r is conventional)')
    print('  lora_dropout   : 0.1')
    print('  target_modules : ["in_proj", "out_proj"]')
    print('  bias           : "none"')
    print('  task_type      : "FEATURE_EXTRACTION"  (no causal LM head)')
    print()
    print('  ⚠️  NOTE: Kuleshov repos appear private. Confirm target_modules')
    print('  by checking https://huggingface.co/plantcad once repos go public.')

print('\n✅ Cell 3 complete')

Attempting to fetch published LoRA configs...


adapter_config.json:   0%|          | 0.00/781 [00:00<?, ?B/s]

  ✅ plantcad/cell_type_specific_acr_plantcad2_small
     r: 8
     lora_alpha: 32
     lora_dropout: 0.1
     target_modules: ['out_proj', 'x_proj', 'in_proj']
     bias: none
     task_type: SEQ_CLS


adapter_config.json:   0%|          | 0.00/781 [00:00<?, ?B/s]

  ✅ plantcad/cross_species_acr_train_on_arabidopsis_plantcad2_small
     r: 8
     lora_alpha: 32
     lora_dropout: 0.1
     target_modules: ['out_proj', 'x_proj', 'in_proj']
     bias: none
     task_type: SEQ_CLS


adapter_config.json:   0%|          | 0.00/781 [00:00<?, ?B/s]

  ✅ plantcad/cross_species_leaf_absolute_expression_plantcad2_small
     r: 8
     lora_alpha: 32
     lora_dropout: 0.1
     target_modules: ['out_proj', 'x_proj', 'in_proj']
     bias: none
     task_type: SEQ_CLS

MAMBA2 BLOCK — LINEAR PROJECTION LAYERS (valid LoRA targets)
(Paper states: LoRA inserted into feedforward/projection layers)

Layer type            Count  Example shape
--------------------------------------------------
  in_proj                48  torch.Size([3224, 768])
  out_proj               48  torch.Size([768, 1536])

RECOMMENDED LoRA CONFIG for YeastCaduceus

  Source: fetched from Kuleshov HF repo
  r              : 8
  lora_alpha     : 32
  lora_dropout   : 0.1
  target_modules : ['out_proj', 'x_proj', 'in_proj']

✅ Cell 3 complete


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 4 — Data Audit
#
# Verifies the Shorkie GCS download is complete and inspects FASTA format.
# Key questions:
#   1. File counts and sizes as expected?
#   2. Are FASTAs already soft-masked (lowercase = repeats)?
#      → If yes: RepeatMasker not needed
#      → If no: must run before preprocessing
#   3. BigWig track count — should be ~5,215 per Shorkie paper
# ─────────────────────────────────────────────────────────────────────────────

from google.colab import drive
drive.mount('/content/drive')

import os
import subprocess

BASE = '/content/drive/MyDrive/shorkie/data/backup'

targets = {
    'genomes/R64':                   'FASTA — S. cerevisiae reference (used for supervised fine-tuning)',
    'genomes/165_Saccharomycetales': 'FASTA — domain adaptation corpus (MLM pretraining)',
    'genomes/80_strains':            'FASTA — S. cerevisiae strain diversity',
    'supervised/bigwigs':            'BigWig — RNA-seq + ChIP coverage targets (~5,215 expected)',
}

print('SHORKIE DATA INVENTORY')
print('='*60)

audit_results = {}
for label, desc in targets.items():
    path = f'{BASE}/{label}'
    all_files = [(root, f) for root, _, files in os.walk(path) for f in files]
    if all_files:
        total_size = sum(os.path.getsize(f'{r}/{f}') for r, f in all_files)
        exts = sorted(set(f.rsplit('.', 1)[-1] for _, f in all_files))
        status = '✅'
    else:
        total_size = 0
        exts = []
        status = '❌ EMPTY'

    audit_results[label] = {'n': len(all_files), 'gb': total_size/1e9, 'exts': exts}
    print(f'\n{status} {label}')
    print(f'   {desc}')
    print(f'   Files : {len(all_files)} | Size: {total_size/1e9:.2f} GB | Extensions: {exts}')
    for _, f in all_files[:3]:
        print(f'   {f}')
    if len(all_files) > 3:
        print(f'   ... ({len(all_files)-3} more)')

# ── FASTA spot-check: soft-masking and format ─────────────────────────────────
print(f'\n{"="*60}')
print('FASTA SPOT-CHECK (165_Saccharomycetales — first file)')

fasta_dir = f'{BASE}/genomes/165_Saccharomycetales'
first_fasta = next((f for _, _, fs in os.walk(fasta_dir) for f in fs), None)

if first_fasta:
    fpath = f'{fasta_dir}/{first_fasta}'
    print(f'File   : {first_fasta}')

    # Handle gzipped or plain
    if first_fasta.endswith('.gz'):
        result = subprocess.run(['zcat', fpath], capture_output=True, text=True)
        lines = result.stdout.splitlines()[:10]
    else:
        result = subprocess.run(['head', '-10', fpath], capture_output=True, text=True)
        lines = result.stdout.splitlines()

    for line in lines:
        print(f'  {line}')

    # Check soft masking
    seq_lines = [l for l in lines if not l.startswith('>')]
    has_lower = any(c.islower() for l in seq_lines for c in l)
    is_gzipped = first_fasta.endswith('.gz')
    chrom_count = sum(1 for l in lines if l.startswith('>'))

    print()
    print(f'  Gzipped          : {is_gzipped}')
    print(f'  Headers in first 10 lines: {chrom_count}')
    print(f'  Soft-masked (lowercase)  : {has_lower}')
    print(f'  → RepeatMasker needed    : {not has_lower}')
else:
    print('  ❌ No files found in 165_Saccharomycetales')

# ── Final go / no-go ─────────────────────────────────────────────────────────
print(f'\n{"="*60}')
print('GO / NO-GO')

checks = {
    'R64 genome present':         audit_results.get('genomes/R64', {}).get('n', 0) > 0,
    '165 Saccharomycetales present': audit_results.get('genomes/165_Saccharomycetales', {}).get('n', 0) > 0,
    'BigWigs present':            audit_results.get('supervised/bigwigs', {}).get('n', 0) > 0,
}

for check, result in checks.items():
    print(f'  {"✅" if result else "❌"} {check}')

if all(checks.values()):
    print('\n✅ Cell 4 PASSED — data intact, ready for Phase 1 preprocessing')
else:
    print('\n❌ Cell 4 FAILED — missing data, re-run download script')

Mounted at /content/drive
SHORKIE DATA INVENTORY

✅ genomes/R64
   FASTA — S. cerevisiae reference (used for supervised fine-tuning)
   Files : 1 | Size: 0.01 GB | Extensions: ['softmask']
   GCA_000146045_2.cleaned.fasta.masked.dust.softmask

✅ genomes/165_Saccharomycetales
   FASTA — domain adaptation corpus (MLM pretraining)
   Files : 165 | Size: 2.11 GB | Extensions: ['softmask']
   GCA_000002515_1.cleaned.fasta.masked.dust.softmask
   GCA_000002545_2.cleaned.fasta.masked.dust.softmask
   GCA_000003835_1.cleaned.fasta.masked.dust.softmask
   ... (162 more)

✅ genomes/80_strains
   FASTA — S. cerevisiae strain diversity
   Files : 80 | Size: 1.02 GB | Extensions: ['softmask']
   GCA_000146045_2.cleaned.fasta.masked.dust.softmask
   GCA_000662435_2.cleaned.fasta.masked.dust.softmask
   GCA_000975585_2.cleaned.fasta.masked.dust.softmask
   ... (77 more)

✅ supervised/bigwigs
   BigWig — RNA-seq + ChIP coverage targets (~5,215 expected)
   Files : 5249 | Size: 110.53 GB | Extensions: 

In [ ]:
# Check if soft-masking exists anywhere in a file (sample 10k chars from middle)
import os
fpath = '/content/drive/MyDrive/shorkie/data/backup/genomes/165_Saccharomycetales/GCA_000002515_1.cleaned.fasta.masked.dust.softmask'
size = os.path.getsize(fpath)
with open(fpath) as f:
    f.seek(size // 2)  # jump to middle of file
    chunk = f.read(10000)
has_lower = any(c.islower() for c in chunk if c.isalpha())
print(f'Soft-masked: {has_lower}')  # expect True

Soft-masked: True
